# Calendar Dimension Table

Creates a standard calendar dimension table (`dim_calendar`) in the `retail_q.retail_gold` schema. The table includes commonly used date attributes such as year, quarter, month, week, and day-level fields. Start and end dates are configurable via widgets.

In [0]:
# Define the date range for the calendar table using widgets
# Adjust these values to control how much data is populated

dbutils.widgets.text("start_date", "2024-01-01", "Start Date (YYYY-MM-DD)")
dbutils.widgets.text("end_date", "2026-12-31", "End Date (YYYY-MM-DD)")

# Fallback defaults keep job runs working even when widget values are unavailable
start_date = dbutils.widgets.get("start_date") or "2024-01-01"
end_date = dbutils.widgets.get("end_date") or "2026-12-31"

print(f"Calendar table will be populated from {start_date} to {end_date}")

In [0]:
# Create dim_calendar using spark.sql with Python-interpolated dates.
# %sql ${widget} interpolation resolves to empty strings in job runs,
# which breaks to_date() — reading the values in Python is reliable.
spark.sql(f"""
CREATE OR REPLACE TABLE retail_q.retail_gold.dim_calendar AS
WITH date_sequence AS (
  SELECT explode(sequence(
    to_date('{start_date}'),
    to_date('{end_date}'),
    interval 1 day
  )) AS date
)
SELECT
  date                                                                              AS date,

  EXTRACT(YEAR  FROM date)                                                          AS year,

  EXTRACT(QUARTER FROM date)                                                        AS quarter,
  CONCAT('Q', EXTRACT(QUARTER FROM date))                                           AS quarter_name,
  CONCAT(EXTRACT(YEAR FROM date), '-Q', EXTRACT(QUARTER FROM date))                 AS year_quarter,

  EXTRACT(MONTH  FROM date)                                                          AS month,
  DATE_FORMAT(date, 'MMMM')                                                          AS month_name,
  DATE_FORMAT(date, 'MMM')                                                           AS month_abbr,
  DATE_FORMAT(date, 'yyyy-MM')                                                       AS year_month,

  WEEKOFYEAR(date)                                                                   AS week_of_year,
  CEIL(DAYOFMONTH(date) / 7.0)                                                       AS week_of_month,

  DAYOFYEAR(date)                                                                    AS day_of_year,
  DAYOFMONTH(date)                                                                   AS day_of_month,

  DAYOFWEEK(date)                                                                    AS day_of_week,
  DATE_FORMAT(date, 'EEEE')                                                          AS day_name,
  DATE_FORMAT(date, 'EEE')                                                           AS day_abbr,

  DAYOFWEEK(date) IN (1, 7)                                                          AS is_weekend,

  date = date_trunc('month', date)                                                   AS is_first_day_of_month,
  date = last_day(date)                                                              AS is_last_day_of_month,
  date = date_trunc('year', date)                                                     AS is_first_day_of_year,
  date = last_day(date_trunc('year', date))                                           AS is_last_day_of_year
FROM date_sequence
ORDER BY date
""")

In [0]:
%sql
SELECT count(*) AS total_rows, min(date) AS min_date, max(date) AS max_date
FROM retail_q.retail_gold.dim_calendar;

In [0]:
%sql
SELECT * FROM retail_q.retail_gold.dim_calendar
LIMIT 15;